# Notebook 1 — Custom ResNet from Scratch (Keras Subclassing API)

**Model 1 of 3 — owned by: <MEMBER NAME>** · Group 6 · Formative 2

This notebook owns the group's **hand-implemented residual network**
(assignment §3.1): a from-scratch CNN built with the Keras **Subclassing
API** — custom `ResidualBlock(layers.Layer)` and `CustomResNet(keras.Model)`
subclasses with an explicit `call()` — following He et al. (2016),
*Deep Residual Learning for Image Recognition*. It is **not** a pretrained
ResNet; every weight starts random.

**Why a residual design for malaria cells:** infected cells differ from
uninfected ones by small, local texture changes (parasite stains inside the
red blood cell). Deep plain CNNs degrade on such tasks because gradients
vanish through many layers; identity shortcuts let gradients bypass each
block, so deeper stacks stay trainable (He et al. 2016, §3.2). Our design:
stem → 3 residual stages (64/128/256 channels) → GAP head, with 1×1
**projection shortcuts** wherever a shape change occurs.

**Notebook map:** §2 data & group-identical split → §3 architecture &
subclassing proof → §4 the 7-experiment campaign → §6 smoke run →
§7 final model → §8 four required plots → §9 Grad-CAM → §10 error
analysis → §11 results table → logbook.


In [ ]:
# ---------------------------------------------------------------------------
# ONE-TIME SETUP — run this cell first in a fresh Colab session.
# It pulls the group's verified helper code from GitHub and prepares the
# environment. Everything later imports from `common`.
# ---------------------------------------------------------------------------
import os, sys, subprocess, pathlib

pathlib.Path("figures").mkdir(exist_ok=True)   # report figures land here

REPO_URL = "https://github.com/k-nizy/Malaria-CNN-Classification-Project-.git"
if not pathlib.Path("common").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    for item in ["common", "tests"]:
        src = pathlib.Path("Malaria-CNN-Classification-Project-") / item
        if src.exists():
            src.rename(item)
if str(pathlib.Path.cwd()) not in sys.path:
    sys.path.insert(0, str(pathlib.Path.cwd()))

# --- Google Drive: persist dataset + TensorBoard logs across Colab resets ---
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    os.environ["TB_LOGDIR"] = "/content/drive/MyDrive/malaria_project/logs/fit"
    DRIVE_DATA = "/content/drive/MyDrive/malaria_project/data"
else:  # local machine / pytest — everything under ./
    os.environ.setdefault("TB_LOGDIR", "logs/fit")
    DRIVE_DATA = "data"

import tensorflow as tf
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ---------------------------------------------------------------------------
# 1. Imports & reproducibility
# ---------------------------------------------------------------------------
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from common.data import (SEED, IMG_SIZE, BATCH_SIZE, LABEL_MAP,
                         get_data, build_datasets, collect_labels,
                         split_fingerprint, make_synthetic_cell_dataset)
from common.eval import (evaluate_model, evaluate_metrics, Metrics,
                         collect_predictions,
                         final_evaluation_figure, plot_training_curves,
                         plot_confusion_matrix, plot_roc_curve,
                         find_misclassifications, plot_misclassifications)
from common.tb import ExperimentLogger, make_run_name, zip_logs_for_drive

tf.keras.utils.set_random_seed(SEED)   # seeds python/numpy/TF together

MODEL_NAME  = "custom_resnet"     # prefix for every TensorBoard run name
MEMBER_NAME = "<MEMBER 1 NAME>"
print(f"Member notebook ready: {MODEL_NAME} ({MEMBER_NAME}), seed={SEED}")

### 2.1 Why the fingerprint matters

All three members run §2 and must get **byte-identical fingerprints**.
If two notebooks trained on different splits, the group's model comparison
in the report would be invalid — the rubric explicitly demands one
coherent comparison. `build_datasets` splits **by file** (70/15/15, seed
42) *before* shuffling batches, so every image appears in exactly one
split. `get_data` caches the 700 MB NIH zip in Drive, so later sessions
skip the download.


In [ ]:
# ---------------------------------------------------------------------------
# 2. Data: download once, cache in Drive, build the GROUP-IDENTICAL split.
# All three members run this cell and must print the same fingerprint.
# ---------------------------------------------------------------------------
cell_images = get_data(drive_data_dir=DRIVE_DATA)      # 27,558 NIH images

train_ds, val_ds, test_ds = build_datasets(
    cell_images, img_size=IMG_SIZE, batch_size=BATCH_SIZE,
    validation_fraction=0.30, seed=SEED, augment=False)

print("train fingerprint:", split_fingerprint(train_ds))
print("val   fingerprint:", split_fingerprint(val_ds))
print("test  fingerprint:", split_fingerprint(test_ds))

### 2.2 Dataset understanding (EDA)

On Colab, extend this section with: class counts (`Parasitized` vs
`Uninfected`, expect 13,779 each), a grid of random cells from each class,
image size/histogram stats, and a note on artifacts visible in the raw
set (the NIH library images include some irregular/funky cells — worth
flagging before Grad-CAM interpretation, since background/platelet
artifacts can attract attention).

Locally, the synthetic demo below keeps the whole notebook executable:
dark blob = parasite signal, same 1/0 convention as the real labels.


In [ ]:
# Peek at one batch — label convention check BEFORE any training:
# 1 = Parasitized (infected, positive class), 0 = Uninfected.
for x, y in train_ds.take(1):
    print("batch:", x.shape, "labels:", y.numpy().ravel()[:8])
    fig, axes = plt.subplots(1, 6, figsize=(12, 2.2))
    for ax, img, lab in zip(axes, x[:6], y[:6]):
        ax.imshow(img.numpy().astype("uint8"))
        ax.set_title("Parasitized" if int(lab) == 1 else "Uninfected", fontsize=9)
        ax.axis("off")
    plt.show()

In [ ]:
# Local dry run (no Colab needed): a tiny synthetic stand-in with the same
# folder layout and label convention, so the FULL flow below executes
# end-to-end even before the real dataset is downloaded.
_synth = make_synthetic_cell_dataset("data/_demo", per_class=12,
                                     img_size=(64, 64), seed=SEED)
demo_train, demo_val, demo_test = build_datasets(
    _synth, img_size=(64, 64), batch_size=8, seed=SEED)
print("demo split ready — same pipeline as the real dataset")

### 3.1 Architecture decisions and their justification

| Decision | Choice | Justification |
|---|---|---|
| Depth | 3 stages × 2 residual blocks | small 128×128 crops; deeper overfits 27k images quickly |
| Width | 32→64→128→256 | He et al.'s pattern, scaled down; keeps params < 3 M for Colab GPU time |
| Shortcut | identity when shapes match, 1×1 projection otherwise | He et al. 2016 §3.2 option B — required whenever stride or channels change |
| Normalization | BN after every conv | stabilizes from-scratch training; momentum is an experiment lever (exp 02) |
| Head | GAP → Dense(256) → Dropout → Dense(1, sigmoid) | GAP kills position overfitting; single sigmoid = binary diagnosis |
| Loss | binary cross-entropy | balanced classes; probability output feeds ROC-AUC |

**Grad-CAM readiness (assignment §5.4):** `call()` stores the last
convolutional feature map on `self._last_conv_features`, and
`common/gradcam.py` reads exactly that tensor under `GradientTape`.
The §5.4 note warns that encapsulated models fail here — ours cannot,
because the tap is part of the design (verified by
`tests/test_models_gradcam.py::test_gradcam_heatmap_localizes_blob`).


In [ ]:
# ---------------------------------------------------------------------------
# 3. The custom ResNet — Keras Subclassing API (assignment §3.1).
# Implemented in common/models.py and verified by tests; shown here for
# the architecture story (notebook ownership = understand every line).
# ---------------------------------------------------------------------------
from common.models import CustomResNet, ResidualBlock, build_custom_resnet

def make_resnet(spec: dict):
    """Build one experiment variant. `spec` maps straight onto the
    architecture/training levers we chose to study."""
    return build_custom_resnet(
        img_size=IMG_SIZE,
        dropout=spec.get("dropout", 0.3),
        l2=spec.get("l2", 0.0),
        width_mult=spec.get("width_mult", 1.0),
        blocks_per_stage=spec.get("blocks_per_stage", 2),
        bn_momentum=spec.get("bn_momentum", 0.99),
    )

model = make_resnet({})
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy", metrics=["accuracy"])
print(f"CustomResNet parameters: {model.count_params():,}")

The printed checks are the evidence the rubric's §3.1 asks for: a genuine
`keras.Model` **subclass** (not Sequential, not `applications.ResNet50`),
with an explicit `call()`, real residual blocks, and projection shortcuts
wherever shapes change. Parameter count goes in the report's methodology
table alongside the two transfer-learning models.


In [ ]:
# Proof that this is a genuine subclassing-API model (rubric §3.1):
print("is keras.Model subclass :", isinstance(model, tf.keras.Model))
print("is Sequential           :", isinstance(model, tf.keras.Sequential))
print("has explicit call()     :", callable(getattr(CustomResNet, "call")))
print("residual blocks         :", len(model.stage1 + model.stage2 + model.stage3))
print("projection shortcuts    :",
      sum(1 for s in (model.stage2, model.stage3) for b in s if b.proj is not None))
print("grad-cam tap populated  :", model._last_conv_features is not None
      or "populated on first forward pass")

### 4.1 Experiment design — what each run tests and why

The campaign is a *progression*, not scattered attempts (rubric §4.1):
capacity first (exp 2–4), then regularization (exp 5–6), then the
combined final recipe (exp 7). Every hypothesis below was written
**before** the run and revisited in the logbook (§13).

| run | variable changed | hypothesis |
|---|---|---|
| exp_01_baseline | — (reference) | establishes the training behaviour we compare against |
| exp_02_bn_momentum_0.9 | BN momentum 0.99→0.9 | faster-moving statistics → smoother val curves |
| exp_03_width_mult_0.5 | half width | the task may not need 2.7 M params; smaller = faster + less overfit |
| exp_04_deeper_augmented | +1 block/stage + augmentation | more capacity only helps once augmentation fights the overfitting it invites |
| exp_05_dropout_0.5 | head dropout 0.3→0.5 | shrinks the train/val gap seen in exp_01 |
| exp_06_l2_1e-4 | L2 on conv kernels | weight decay as an alternative regularizer |
| exp_07_final | best combination @ lr 2e-4 | lower LR for the final polish, combining what worked |

Each run's full configuration is logged in its TensorBoard HPARAMS record
and its `training_log.csv` — the report cites run names, not vibes.


In [ ]:
# ---------------------------------------------------------------------------
# 4. The experiment campaign — 7 documented runs (rubric §4.1: meaningful
# changes only, clear progression from baseline to final).
# ---------------------------------------------------------------------------
EXPERIMENTS = [
    dict(id=1, name="baseline",
         config=dict(),
         lr=1e-3,
         note="Reference run: default width/depth, no augmentation."),
    dict(id=2, name="bn_momentum_0.9",
         config=dict(bn_momentum=0.9),
         lr=1e-3,
         note="Lower BN momentum -> moving statistics track the batch more "
              "closely; tests stability under augmentation-free training."),
    dict(id=3, name="width_mult_0.5",
         config=dict(width_mult=0.5),
         lr=1e-3,
         note="Half-width network: how much capacity does the task need?"),
    dict(id=4, name="deeper_augmented",
         config=dict(blocks_per_stage=3, augment=True),
         lr=1e-3,
         note="Extra residual block per stage + flip/rot/zoom augmentation."),
    dict(id=5, name="dropout_0.5",
         config=dict(dropout=0.5),
         lr=1e-3,
         note="Stronger head regularisation for the overfitting gap."),
    dict(id=6, name="l2_1e-4",
         config=dict(l2=1e-4),
         lr=1e-3,
         note="Weight decay on every convolution kernel (He et al. recipe)."),
    dict(id=7, name="final_lr_2e-4",
         config=dict(dropout=0.3, l2=1e-4, augment=True),
         lr=2e-4,
         note="Best-of combination at a lower learning rate."),
]
assert len(EXPERIMENTS) >= 7
for e in EXPERIMENTS:
    print(make_run_name(MODEL_NAME, e["id"], e["name"]))

### 5.1 What the runner guarantees

One function = one experiment, so nothing drifts between runs. Each run
writes: (1) a TensorBoard directory named
`{model}_exp_{id}_{description}` — the rubric's naming convention,
enforced by `make_run_name`; (2) an HPARAMS record (lr, optimizer, batch
size, epochs, augmentation, dropout, L2, strategy) for the hparams
dashboard comparison screenshots; (3) `training_log.csv` as extra
evidence; (4) final test metrics *into the same run directory* via
`log_final_metrics`, so run-to-run comparisons in TensorBoard include
the end-of-training evaluation.

On Colab, `TB_LOGDIR` points into Drive (set in §0), so a disconnected
session cannot eat 30 epochs of evidence.


In [ ]:
# ---------------------------------------------------------------------------
# 5. The experiment runner — one function, used for all 7 runs.
# Every run writes a TensorBoard directory whose NAME carries the experiment
# id and description, and whose HPARAMS record carries the configuration —
# so every claim in the report is traceable to a logged run (rubric §4.2).
# ---------------------------------------------------------------------------
from common.models import build_custom_resnet  # direct import for the runner

def run_experiment(spec: dict, datasets=None, epochs: int = 30,
                   logdir: str | None = None, verbose: int = 1):
    """Train one experiment variant and log everything the rubric requires."""
    train_ds, val_ds, test_ds = datasets

    exp = ExperimentLogger(
        model_name=MODEL_NAME, exp_id=spec["id"], description=spec["name"],
        hparams=dict(lr=spec["lr"], optimizer="adam",
                     batch_size=BATCH_SIZE, epochs=epochs,
                     augmentation=spec.get("augment", False),
                     dropout=spec.get("dropout", 0.3),
                     l2=spec.get("l2", 0.0),
                     width_mult=spec.get("width_mult", 1.0),
                     blocks_per_stage=spec.get("blocks_per_stage", 2),
                     bn_momentum=spec.get("bn_momentum", 0.99),
                     strategy="scratch",                       # from-scratch
                     model="CustomResNet"),
        logdir=logdir)

    augment = spec.get("augment", False)
    if augment:
        # rebuild only the train split with augmentation enabled
        tr, va, te = build_datasets(cell_images, img_size=IMG_SIZE,
                                    batch_size=BATCH_SIZE, seed=SEED,
                                    augment=True)
        train_ds = tr
    model = make_resnet(spec)
    model.compile(optimizer=tf.keras.optimizers.Adam(spec["lr"]),
                  loss="binary_crossentropy", metrics=["accuracy"])

    history = model.fit(train_ds, validation_data=val_ds, epochs=epochs,
                        callbacks=exp.callbacks(), verbose=verbose)

    metrics, y_true, y_prob = evaluate_model(model, test_ds)
    exp.log_final_metrics(metrics.as_row())
    exp.log_hparams_once(metrics.as_row())
    print(f"[{{exp.run_name}}] {{metrics.format()}}")
    return dict(run=exp.run_name, model=model, history=history,
                metrics=metrics, y_true=y_true, y_prob=y_prob,
                spec=spec, note=spec["note"])


def result_to_row(r: dict) -> dict:
    """One row for the report's per-experiment metrics table (§5.1)."""
    row = dict(experiment=r["run"])
    row.update(r["metrics"].as_row())
    return row

### 6.1 Reading the smoke run

Three demo epochs on 24 synthetic images should already beat 0.5 val
accuracy — the blob signal is easy. What we are really checking:

1. **No crashes in the full glue** — builder → fit → evaluate → log.
2. **TensorBoard artifacts exist** — the listing proves where evidence
   lives before the real campaign runs.
3. **Metrics row is complete** — the JSON line is exactly one row of the
   report's §5.1 table (accuracy, precision, recall/sensitivity,
   specificity, F1, ROC-AUC).

Expected on the real dataset: exp_01 baseline lands around the high-80s
accuracy after ~30 epochs (published custom CNNs on this exact NIH set
reach 89–97% sensitivity); our target bar for custom from-scratch models
is ≥ 0.90 sensitivity and ≥ 0.90 specificity (assignment §5.1).


In [ ]:
# ---------------------------------------------------------------------------
# 6. Smoke run — the FULL loop on the tiny demo dataset (seconds, not hours).
# Confirms: builder -> fit -> evaluate -> TensorBoard artifacts, before we
# spend GPU hours on the real campaign. On Colab with the real data, replace
# `datasets=demo` with `datasets=(train_ds, val_ds, test_ds)` and epochs=30.
# ---------------------------------------------------------------------------
demo = (demo_train, demo_val, demo_test)
demo_spec = dict(id=0, name="demo_smoke", config=dict(width_mult=0.25), lr=1e-3)
demo_run = run_experiment(demo_spec, datasets=demo, epochs=3, verbose=1)

_demo_dir = Path(os.environ["TB_LOGDIR"]) / demo_run["run"]
print("TensorBoard artifacts written:")
for p in sorted(_demo_dir.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(_demo_dir))
print("\nMetrics row for the report table:")
import json
print(json.dumps(result_to_row(demo_run), indent=2, default=str))

### 7.1 From experiments to the final model

The final version is exp_07's recipe retrained with the full epoch budget
and the learning-rate drop that late-training curves justified. In a real
session this cell loads the checkpoint of the best run and (optionally)
retrains with more epochs. `print_history` gives the epoch-by-epoch
numbers the report prose quotes — e.g. "validation loss plateaued at
epoch N near X" must be traceable to this table *and* the TensorBoard
scalars for the same run.


In [ ]:
# ---------------------------------------------------------------------------
# 7. FINAL MODEL — retrain the winning configuration with the full budget
# (rubric §5.2 visualizations are for the final version of each model).
# On Colab: pick the best experiment id from section 6 and run e.g.
#     final = run_experiment(EXPERIMENTS[6], datasets=(train_ds, val_ds, test_ds),
#                            epochs=40, logdir=os.environ["TB_LOGDIR"])
# Here we execute the glue with the demo-scale data so the notebook is
# runnable end-to-end; history is loaded from a checkpoint in a real run.
# ---------------------------------------------------------------------------
final_spec = EXPERIMENTS[6]
final = run_experiment(dict(final_spec, name="final"), datasets=demo,
                       epochs=3, verbose=1)

def print_history(history):
    """Epoch-by-epoch summary — the numbers quoted in the report prose."""
    for i, (l, a, vl, va) in enumerate(zip(history.history["loss"],
                                           history.history["accuracy"],
                                           history.history["val_loss"],
                                           history.history["val_accuracy"]), 1):
        print(f"epoch {{i:02d}}  loss={{l:.4f}}  acc={{a:.4f}}  "
              f"val_loss={{vl:.4f}}  val_acc={{va:.4f}}")

print_history(final["history"])

### 8.1 Interpreting the four plots (write this in your own words after
the real run — prompts, not final prose)

- **Accuracy curve** — does validation accuracy track training, or split
  apart (overfitting)? Where does it plateau?
- **Loss curve** — the honest plot: loss exposes what accuracy hides.
  A rising val loss with flat val accuracy = memorization starting.
- **Confusion matrix** — for malaria, the top-left vs top-right asymmetry
  *is* the story: false negatives (infected → uninfected) are the costly
  error. State FN and FP counts explicitly.
- **ROC curve** — threshold-free view of the same trade-off. The AUC
  number in §11 and this curve are the same evidence at two zoom levels;
  explain what moving the operating threshold does to sensitivity vs
  specificity, and where WHO-style priorities would put it.


In [ ]:
# ---------------------------------------------------------------------------
# 8. Final evaluation: the rubric's four plots in ONE row + full metrics.
# ---------------------------------------------------------------------------
fig, m, y_true, y_prob = final_evaluation_figure(
    final["model"], final["history"], demo_test,
    save_path="figures/" + MODEL_NAME + "_final_evaluation.png")
plt.show()
print(m.format())
print("\nWHO naming:  sensitivity = recall(infected) ="
      f" {{m.sensitivity:.4f}}   specificity = {{m.specificity:.4f}}")
print("Targets — custom from-scratch: sensitivity >= 0.90, specificity >= 0.90")

### 9.1 Interpreting the heatmaps (prompts — answer after the real run)

- Does the pretrained backbone attend **inside the cell** or to
  background/stain? ImageNet features sometimes latch onto global color
  statistics — that would be a shortcut worth flagging.
- Compare against the custom ResNet notebook: pretrained features often
  produce sharper, more object-like attention; a from-scratch model may
  spread attention. Explain what that means for trust.
- The **incorrect** example: did attention follow the artifact that
  fooled the model? Grad-CAM on errors is how the rubric separates
  description from analysis.


In [ ]:
# ---------------------------------------------------------------------------
# 9. Explainability (rubric §5.4): Grad-CAM on the subclassed model.
# common/gradcam.py implements the eager GradientTape path that reads the
# feature map our model stashes on `model._last_conv_features` — the exact
# §5.4 requirement that encapsulated models fail. Heatmaps are computed on
# the LOGIT (pre-sigmoid) because a saturated sigmoid kills gradients.
# ---------------------------------------------------------------------------
from common.gradcam import (make_gradcam_heatmap, overlay_heatmap,
                            pick_gradcam_examples, gradcam_report_grid)

y_true_all, y_prob_all = collect_predictions(final["model"], demo_test)
images_all = np.concatenate([x.numpy() for x, _ in demo_test])

examples = pick_gradcam_examples(y_true_all, y_prob_all, images_all)
print("picked:", ["correct-infected", "correct-uninfected", "incorrect"][:len(examples)])

fig = gradcam_report_grid(examples, final["model"],
                          save_path="figures/" + MODEL_NAME + "_gradcam.png")
plt.show()

### 10.1 Error analysis — how to write it up

For each mined misclassification: **true vs predicted label + the model's
confidence**, then a plausible cause (stain artifact, late-stage vs early
parasite, ambiguous cell, background clutter). The `diagnose_fit` readout
is deliberately simple — the report should cite the *specific* curve
evidence (epoch numbers, gap sizes) and what was tried in response
(which experiment id came out of which diagnosis).


In [ ]:
# ---------------------------------------------------------------------------
# 10. Error analysis (rubric §5.3): misclassifications + fit diagnosis.
# ---------------------------------------------------------------------------
imgs, trues, preds, probs = find_misclassifications(final["model"], demo_test, n=3)
fig = plot_misclassifications(imgs, trues, preds, probs)
plt.show()

def diagnose_fit(history):
    """Simple overfitting/underfitting read-out from the curves."""
    h = history.history
    best = int(np.argmin(h["val_loss"]))
    gap = h["accuracy"][-1] - h["val_accuracy"][-1]
    verdict = []
    if gap > 0.05:
        verdict.append(f"OVERFITTING sign: train-val accuracy gap {{gap:.3f}}")
    if min(h["val_loss"]) > 0.693:
        verdict.append("UNDERFITTING sign: val_loss never beat the 0.693 chance line")
    if best < len(h["val_loss"]) - 2:
        verdict.append(f"early stop candidate: best val epoch {{best + 1}}")
    return "; ".join(verdict) or "no strong over/underfitting signal"

print("diagnosis:", diagnose_fit(final["history"]))

### 11. Results table & TensorBoard evidence

`scan_runs()` rebuilds the per-experiment metrics table from the run
folders themselves — the same numbers TensorBoard shows, in CSV-ready
form. Paste it into the report (one table per model, one row per
experiment, run names exactly as logged). For the appendix: export the
Drive `logs/fit` directory (zip helper below) and screenshot the hparams
dashboard comparison — that is the rubric's "TensorBoard evidence showing
run-to-run comparisons".


In [ ]:
# ---------------------------------------------------------------------------
# 11. Per-experiment table straight from the TensorBoard run folders —
# paste-ready for the report and a backup if TensorBoard is unavailable.
# ---------------------------------------------------------------------------
import csv

def scan_runs(logdir=None):
    logdir = Path(logdir or os.environ["TB_LOGDIR"])
    rows = []
    for d in sorted(logdir.iterdir()):
        f = d / "final_metrics.txt"
        if f.exists():
            row = dict(run=d.name)
            row.update(line.split(": ", 1) for line in f.read_text().splitlines())
            rows.append(row)
    return rows

for row in scan_runs():
    print(row)
# with open("results_table.csv", "w", newline="") as fh:
#     w = csv.DictWriter(fh, fieldnames=list(scan_runs()[0].keys())); w.writerows(...)

### 12. Oral defense preparation (10–15 min: 2-min overview + Q&A)

Expect to be asked: *why residual connections here?* (gradient flow,
He et al.), *why this depth/width?* (exp 03/04 evidence), *what did BN
momentum change?* (exp 02 curves), *where does the model look and do you
trust it?* (§9), *what is your model's worst error and why?* (§10),
*where does it rank among the group's three models and what evidence says
so?* (§11 table + ROC trade-offs, not accuracy alone).


### 13. Lab journal (write as you go — this becomes the report's voice)

After each experiment, four lines: **what I changed · what I expected ·
what happened · what surprised me**. Example shape (do not copy):

> exp_03 (width 0.5): expected a small accuracy drop; got −0.8 pts and a
> *tighter* train/val gap. Surprised: half the parameters still beat the
> baseline's validation loss by epoch 12.

Honest negative results are rubric gold: the campaign must read as a
progression of *decisions*, and decisions come from expectations that
failed as well as succeeded.
